# RQ1 - Clones (non-agentic vs agentic)

## Preparing data

In [4]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

RESULTS_DIR = Path("../../results")

rows = []

for file in RESULTS_DIR.glob("*_detailed_results.json"):

    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Extract file index
    prefix = int(file.name.split("_")[0])

    rows.append({
        "file": file.name,
        "file_index": prefix,
        "agentic": prefix % 2 == 1, # odd numbers are agentic, even numbers are non-agentic
        "base_index": prefix - 1 if prefix % 2 == 1 else prefix,
        "project": "_".join(file.stem.split("_")[1:-2]),
        "results": data
    })


len(rows)

59

## Summary per project

In [ ]:
expanded = []

for r in rows:

    for entry in r["results"]:

        expanded.append({
            "file": r["file"],
            "file_index": r["file_index"],
            "base_index": r["base_index"],
            "agentic": r["agentic"],
            "project": r["project"],

            "model": entry["model"],
            "language": entry["language"],
            "pair_id": entry["pair_id"],

            "sim": entry["sim"],
            "codebleu": entry["codebleu"],
        })


df = pd.DataFrame(expanded)

print(df.shape)
display(df.head())

## Compare distributions

In [1]:
df.groupby("agentic")[["sim", "codebleu"]].describe()

NameError: name 'df' is not defined

In [ ]:
df.groupby("agentic")[["sim", "codebleu"]].mean()

## Boxplots

### Semantic similarity

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))

df.boxplot(
    column="sim",
    by="agentic",
    ax=ax
)

ax.set_title("Semantic Similarity (SimCSE/Cosine)")
ax.set_xlabel("Agentic")
ax.set_ylabel("Similarity")

plt.suptitle("")
plt.show()

### Syntactic similarity

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))

df.boxplot(
    column="codebleu",
    by="agentic",
    ax=ax
)

ax.set_title("Syntactic Similarity (CodeBLEU)")
ax.set_xlabel("Agentic")
ax.set_ylabel("CodeBLEU")

plt.suptitle("")
plt.show()

## Paired comparison

In [ ]:
project_scores = (
    df
    .groupby(
        [
            "project",
            "base_index",
            "agentic"
        ]
    )[["sim", "codebleu"]]
    .mean()
    .reset_index()
)

display(project_scores.head())

In [ ]:
comparison = project_scores.pivot_table(
    index=["project", "base_index"],
    columns="agentic",
    values=["sim", "codebleu"]
)

comparison.head()

## Evolution

In [ ]:
comparison["sim_change"] = (
    comparison[("sim", True)]
    -
    comparison[("sim", False)]
)

comparison["codebleu_change"] = (
    comparison[("codebleu", True)]
    -
    comparison[("codebleu", False)]
)

comparison.head()